# Piloto de audiolivro adaptado: Aventuras de Sherlock Holmes (A Liga dos Cabeças Vermelhas, caps. 1 a 3; O Carbúnculo Azul, caps. 4 a 6)

Narra, numa GPU grátis do Google Colab, os capítulos 1 a 6 de *Aventuras de Sherlock Holmes*, numa adaptação para crianças e adolescentes com deficiência visual, síndrome de Down, autismo e TDAH. O guia de adaptação e de narração está no `GUIA.md` da pasta Audiolivros.

**Antes de começar:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois rode as células de cima para baixo. Rodar de novo qualquer célula é seguro.

**Saída** (no seu Google Drive, em `MyDrive/Audiolivros/sherlock/capNN/`), para cada capítulo:
- `capNN_A_so_voz.mp3`: só a narração, para quem é sensível a sons;
- `capNN_B_com_ambiente.mp3`: narração com vinheta suave, alguns sons curtos e baixinhos nas pausas (sonoplastia leve) e, nas cenas do riacho, som de água bem baixinho.

Escolha os capítulos na célula 2 (`CAPITULOS_PARA_GERAR`). As vozes ficam em `sherlock/vozes/` e são as mesmas em todos os capítulos.

**Licença:** a voz padrão (OmniVoice) é de uso **não comercial** (CC-BY-NC). Serve para um piloto gratuito. As vozes são sintéticas e originais, sem clonar ninguém.

**Sotaque:** o modelo fala português, mas às vezes puxa para o sotaque de Portugal. Se acontecer, troque a `semente` da voz na célula 5. Outra saída é usar uma gravação de 5 a 10 segundos de alguém que **autorizou por escrito** o uso da própria voz (campo `amostra`).

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("ATENÇÃO: sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
          "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive e pastas ────────────────────────────────────────────────
import os
from google.colab import drive
drive.mount("/content/drive")

CAPITULOS_PARA_GERAR = [1, 2, 3, 4, 5, 6]   # quais capítulos narrar (ex.: [2, 3])

LIVRO = "/content/drive/MyDrive/Audiolivros/sherlock"
PASTA_VOZES = os.path.join(LIVRO, "vozes")   # uma fala de referência por personagem
os.makedirs(PASTA_VOZES, exist_ok=True)
print("Os arquivos vão para:", LIVRO)

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min) ──────────────────────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="instalação do backend")

from huggingface_hub import snapshot_download
print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
snapshot_download("k2-fsa/OmniVoice")
print("Pronto.")

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4.5. Aceitar as licenças dos modelos (obrigatório antes de gerar) ───────
# Sem este passo o /generate falha. Rodar esta célula aceita as licenças em seu nome:
# OmniVoice é CC-BY-NC (uso NÃO comercial: o áudio não pode ser vendido).
import requests
BASE = "http://127.0.0.1:3900"

for repo in ("k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"):
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    if st.get("accepted"):
        print(f"[ok] {repo} já aceito")
        continue
    r = requests.post(
        f"{BASE}/models/licenses/accept",
        json={"repo_id": repo, "fingerprint": st["fingerprint"], "accepted": True},
        timeout=30,
    )
    r.raise_for_status()
    print(f"[aceito] {repo} ({st.get('license')})")

## Roteiro

Cada trecho é `[personagem, texto, velocidade, pausa depois em segundos]` e, às vezes, um quinto item: o **efeito** que toca na pausa, só na versão B (`transicao`, `magia`, `agua`, `galope`, `porta`, `zumbido`, `festa`). Cada capítulo tem `"ambiente"`: `"riacho"` (água baixinha durante a fala) ou `"nenhum"`. Regras em `SONOPLASTIA.md`.

- **velocidade:** 1.0 é normal; até 1.1 na ação; 0.85 a 0.9 no suspense e no sono.
- **pausa:** o silêncio depois do trecho. Pausas maiores antes das revelações prendem a atenção.
- **voz** aceita só estes termos: `male`/`female`; `child`, `teenager`, `young adult`, `middle-aged`, `elderly`; `very low pitch` … `very high pitch`; `whisper`.
- A voz de cada personagem nasce da fala mais longa dele no **primeiro** capítulo em que aparece, e é a mesma em todos os outros.

Edite à vontade. Os trechos que você mudar são gerados de novo na célula 7.

In [ ]:
# ── 5. Roteiro (EDITE AQUI) ─────────────────────────────────────────────────
# "amostra" (opcional, em cada voz): caminho de um .wav de 5-10 s de alguém que
# AUTORIZOU por escrito o uso da voz. Nunca use a voz de um dublador ou artista.
VOZES = {'narrador': {'voz': 'male, young adult, moderate pitch', 'semente': 201},
 'holmes': {'voz': 'male, young adult, low pitch', 'semente': 301},
 'watson': {'voz': 'male, middle-aged, moderate pitch', 'semente': 311},
 'wilson': {'voz': 'male, elderly, moderate pitch', 'semente': 321},
 'ross': {'voz': 'male, middle-aged, high pitch', 'semente': 331},
 'spaulding': {'voz': 'male, young adult, low pitch', 'semente': 341},
 'jones': {'voz': 'male, middle-aged, very low pitch', 'semente': 351},
 'merryweather': {'voz': 'male, elderly, low pitch', 'semente': 361},
 'peterson': {'voz': 'male, middle-aged, high pitch', 'semente': 371},
 'baker': {'voz': 'male, elderly, very low pitch', 'semente': 381},
 'breckinridge': {'voz': 'male, middle-aged, low pitch', 'semente': 391},
 'ryder': {'voz': 'male, young adult, very high pitch', 'semente': 401},
 'maggie': {'voz': 'female, middle-aged, moderate pitch', 'semente': 411}}

CAPITULOS = {
    1: {"titulo": "Um anúncio muito estranho", "ambiente": "nenhum", "trechos": [
        ["narrador", "Sherlock Holmes: A Liga dos Cabeças Vermelhas. Capítulo um: Um anúncio muito estranho.", 0.95, 1.2],
        ["narrador", "Esta história acontece em Londres, uma cidade da Inglaterra, há mais de cem anos.", 1.0, 0.5],
        ["narrador", "Naquele tempo, as ruas têm carruagens puxadas por cavalos. À noite, as luzes das ruas são lampiões a gás.", 1.0, 0.6],
        ["narrador", "Numa casa da Rua Baker mora Sherlock Holmes. Holmes é um detetive. Detetive é a pessoa que descobre segredos e resolve mistérios.", 1.0, 0.6],
        ["narrador", "Holmes é alto e magro. Ele presta atenção em tudo, até nas coisas bem pequenas.", 1.0, 0.5],
        ["narrador", "O melhor amigo de Holmes é o doutor Watson. Watson é médico e ajuda Holmes nos mistérios.", 1.0, 0.8],
        ["narrador", "Um dia, Watson chega à casa de Holmes. Na sala, encontra um senhor gordo, de rosto vermelho.", 1.0, 0.5],
        ["narrador", "O cabelo desse senhor é muito, muito vermelho. Vermelho como fogo.", 1.0, 0.6],
        ["holmes", "Watson! Você chegou na hora certa. Este é o senhor Wilson. Ele me trouxe um mistério.", 1.0, 0.6],
        ["narrador", "Holmes olha para o senhor Wilson de cima a baixo. Depois, sorri.", 0.95, 0.6],
        ["holmes", "O senhor já trabalhou com as mãos, carregando coisas pesadas. E já esteve na China.", 1.0, 0.6],
        ["wilson", "Como o senhor sabe disso? Eu não contei nada!", 1.05, 0.6],
        ["holmes", "A sua mão direita é maior que a esquerda. Isso acontece com quem faz força com essa mão por muitos anos.", 1.0, 0.5],
        ["holmes", "E no seu pulso tem a tatuagem de um peixe cor-de-rosa. Esse tipo de tatuagem só se faz na China.", 1.0, 0.7],
        ["narrador", "Viu só? Holmes não adivinha. Ele olha as pistas e pensa no porquê de cada uma.", 1.0, 0.8],
        ["narrador", "O senhor Wilson fica mais calmo e começa a contar a história dele.", 1.0, 0.5],
        ["wilson", "Eu tenho uma lojinha pequena. Ela fica numa praça chamada Saxe-Coburg.", 1.0, 0.5],
        ["wilson", "Tenho um ajudante, o Vicente Spaulding. Ele trabalha muito bem e quer ganhar só metade do salário.", 1.0, 0.6],
        ["narrador", "Lembre desse detalhe, porque ele é importante: o ajudante quer ganhar só a metade. Um pouco estranho, não é?", 0.95, 0.9],
        ["wilson", "Um dia, o Spaulding chegou correndo com um jornal na mão.", 1.05, 0.4],
        ["spaulding", "Senhor Wilson! Olhe este anúncio! A Liga dos Cabeças Vermelhas está procurando um homem de cabelo vermelho!", 1.05, 0.6],
        ["narrador", "Liga quer dizer um grupo de pessoas. E esse grupo só aceita gente de cabelo vermelho.", 1.0, 0.5],
        ["narrador", "O anúncio diz assim: a Liga paga quatro libras por semana para um trabalho bem fácil. Naquele tempo, era muito dinheiro.", 1.0, 0.6],
        ["narrador", "O senhor Wilson precisa de dinheiro, porque a loja vende pouco. Por isso, ele vai até o endereço do anúncio.", 1.0, 0.6],
        ["narrador", "Lá, a rua está cheia de homens de cabelo vermelho. Tem cabelo cor de laranja, cor de tijolo, cor de cenoura.", 1.05, 0.5],
        ["narrador", "Mas nenhum cabelo é tão vermelho quanto o do senhor Wilson.", 1.0, 0.6],
        ["narrador", "Num escritório pequeno, espera um homem também de cabelo vermelho. O nome dele é Duncan Ross.", 1.0, 0.5],
        ["ross", "Que cabelo maravilhoso! O emprego é seu.", 1.05, 0.5],
        ["narrador", "Para ter certeza, o senhor Ross puxa o cabelo do senhor Wilson com as duas mãos. Com força!", 1.05, 0.4],
        ["wilson", "Ai!", 1.1, 0.4],
        ["ross", "Desculpe. Eu precisava ver se o cabelo era de verdade, e não uma peruca.", 1.0, 0.7],
        ["narrador", "Até agora: o senhor Wilson tem cabelo vermelho e ganhou um emprego fácil e bem pago. Mas por que alguém pagaria tanto por um trabalho tão fácil?", 0.95, 1.2],
        ["ross", "O trabalho é copiar uma enciclopédia. Enciclopédia é um livro enorme, que explica as coisas do mundo, de A a Z.", 1.0, 0.5],
        ["ross", "O senhor vem todo dia, das dez da manhã às duas da tarde. E não pode sair da sala de jeito nenhum. Se sair, perde o emprego.", 1.0, 0.7],
        ["narrador", "O senhor Wilson aceita. Ele sabe que o ajudante Spaulding pode cuidar da loja enquanto ele está fora.", 1.0, 0.6],
        ["narrador", "E assim, todo dia, ele escreve, escreve e escreve. Copia palavras com a letra A: abelha, arco, armadura.", 1.0, 0.6],
        ["narrador", "Todo sábado, ele recebe as quatro libras. O senhor Wilson está muito feliz.", 1.0, 0.6],
        ["narrador", "Passam oito semanas. Ele já encheu muitas folhas de papel.", 0.95, 0.8],
        ["narrador", "O que você acha que vai acontecer com esse emprego tão bom?", 0.95, 2.5],
        ["narrador", "Para lembrar: o senhor Wilson copia uma enciclopédia todos os dias, longe da loja, e ganha bem por isso.", 0.95, 0.8],
        ["narrador", "Até que, numa manhã, ele chega ao escritório e a porta está trancada. Na porta, tem um pequeno cartão pregado.", 0.9, 0.6],
        ["narrador", "No cartão está escrito: a Liga dos Cabeças Vermelhas acabou.", 0.85, 1.6],
        ["narrador", "Para onde foi a Liga? E por que ela sumiu? Isso fica para o próximo capítulo. Até lá!", 1.0, 0.8],
    ]},
    2: {"titulo": "As pistas na praça", "ambiente": "nenhum", "trechos": [
        ["narrador", "Sherlock Holmes: A Liga dos Cabeças Vermelhas. Capítulo dois: As pistas na praça.", 0.95, 1.2],
        ["narrador", "No capítulo passado, o senhor Wilson, de cabelo bem vermelho, ganhou um emprego estranho: copiar uma enciclopédia, longe da loja dele.", 1.0, 0.5],
        ["narrador", "Depois de oito semanas, a porta do escritório amanheceu trancada. A Liga dos Cabeças Vermelhas tinha acabado.", 1.0, 0.8],
        ["narrador", "Agora, o senhor Wilson está na sala de Sherlock Holmes. Ele está bravo e triste.", 1.0, 0.5],
        ["wilson", "Eu perguntei ao dono do prédio. Ele disse que nunca ouviu falar de Duncan Ross nem de Liga nenhuma!", 1.05, 0.5],
        ["wilson", "Eu perdi o meu emprego, senhor Holmes. Quero saber quem fez essa brincadeira comigo, e por quê.", 1.0, 0.7],
        ["holmes", "Vou descobrir, senhor Wilson. Primeiro, me fale mais do seu ajudante, o Spaulding.", 1.0, 0.5],
        ["narrador", "Watson estranha. Por que Holmes quer saber do ajudante, e não da Liga?", 0.95, 0.8],
        ["wilson", "O Spaulding tem uns trinta anos. É baixo, rápido, e tem uma mancha branca na testa.", 1.0, 0.5],
        ["wilson", "Ele gosta muito de tirar fotografias. Passa horas lá embaixo, no porão da loja. Porão é um quarto debaixo da casa.", 1.0, 0.6],
        ["narrador", "Holmes fica animado. Os olhos dele brilham.", 1.0, 0.5],
        ["holmes", "Obrigado, senhor Wilson. Até segunda-feira, eu terei uma resposta.", 1.0, 0.8],
        ["narrador", "O senhor Wilson vai embora. Holmes senta na poltrona, fecha os olhos e fica pensando, quietinho, por muito tempo.", 0.9, 0.6],
        ["narrador", "Watson até acha que o amigo dormiu. Mas não! Holmes pula da poltrona.", 1.05, 0.4],
        ["holmes", "Watson, vamos até a praça do senhor Wilson. Quero ver as coisas com os meus próprios olhos.", 1.05, 0.8],
        ["narrador", "Até agora: a Liga sumiu, o senhor Wilson perdeu o emprego, e Holmes ficou muito interessado no ajudante que passa horas no porão.", 0.95, 1.2],
        ["narrador", "A praça Saxe-Coburg é pequena e cinzenta. As casas são velhas, e a loja do senhor Wilson tem uma placa na porta.", 1.0, 0.6],
        ["narrador", "Holmes faz uma coisa esquisita. Ele bate com a bengala no chão da calçada, na frente da loja. Toc, toc, toc.", 1.0, 0.6],
        ["narrador", "Ele está escutando o som do chão. Sabe por quê? Porque um chão oco, com um buraco embaixo, faz um som diferente.", 0.95, 0.8],
        ["narrador", "Depois, Holmes bate na porta da loja. Quem abre é o ajudante, o Spaulding.", 1.0, 0.5],
        ["holmes", "Bom dia. O senhor pode me ensinar o caminho para uma rua chamada Strand?", 1.0, 0.5],
        ["spaulding", "Terceira rua à direita, quarta à esquerda.", 1.05, 0.5],
        ["narrador", "O ajudante fecha a porta bem rápido. Holmes não queria saber o caminho de verdade. Ele queria olhar para o Spaulding.", 1.0, 0.6],
        ["holmes", "Você viu os joelhos da calça dele, Watson? Estão gastos e sujos de terra.", 1.0, 0.8],
        ["narrador", "Por que alguém teria terra nos joelhos da calça? Pense um pouquinho.", 0.95, 2.5],
        ["narrador", "Depois, os dois amigos viram a esquina e olham a rua de trás da praça.", 1.0, 0.5],
        ["narrador", "Essa rua é bem diferente: cheia de gente, de carruagens e de lojas bonitas. E bem atrás da loja do senhor Wilson, tem um banco.", 1.0, 0.6],
        ["narrador", "Banco é o lugar onde as pessoas guardam dinheiro. E esse banco guarda muito dinheiro.", 1.0, 0.8],
        ["narrador", "Holmes fica sério de repente.", 0.9, 0.5],
        ["holmes", "Watson, um crime vai acontecer hoje à noite. Preciso da sua ajuda. Venha à minha casa às dez horas.", 0.95, 0.8],
        ["watson", "Que crime, Holmes? Eu não entendi nada!", 1.0, 0.6],
        ["holmes", "Hoje à noite, você vai entender. Hoje é sábado, e isso deixa tudo mais urgente.", 0.95, 0.8],
        ["narrador", "Para lembrar: Holmes bateu no chão da calçada, viu terra nos joelhos do ajudante e descobriu um banco atrás da loja.", 0.95, 0.8],
        ["narrador", "Que crime será esse? Isso fica para o próximo capítulo. Até lá!", 1.0, 0.8],
    ]},
    3: {"titulo": "Escuro no porão do banco", "ambiente": "nenhum", "trechos": [
        ["narrador", "Sherlock Holmes: A Liga dos Cabeças Vermelhas. Capítulo três: Escuro no porão do banco.", 0.95, 1.2],
        ["narrador", "No capítulo passado, Holmes encontrou pistas na praça do senhor Wilson: terra nos joelhos do ajudante e um banco bem atrás da loja.", 1.0, 0.5],
        ["narrador", "E Holmes disse: hoje à noite, um crime vai acontecer.", 0.95, 0.8],
        ["narrador", "São dez horas da noite. Na casa de Holmes estão mais dois homens.", 1.0, 0.5],
        ["narrador", "Um é o inspetor Jones, da polícia. O outro é o senhor Merryweather, o chefe do banco.", 1.0, 0.5],
        ["jones", "Senhor Holmes, espero que o senhor tenha razão. Estou atrás de um ladrão muito esperto, o John Clay, há muitos anos.", 1.0, 0.6],
        ["holmes", "Hoje, inspetor, o senhor vai encontrar o John Clay. Eu prometo.", 0.95, 0.7],
        ["narrador", "Os quatro vão de carruagem até o banco. O senhor Merryweather abre uma porta e eles descem uma escada de pedra.", 1.0, 0.5],
        ["narrador", "Lá embaixo fica o porão do banco. É grande, frio e úmido, com cheiro de terra. O chão é de pedras grandes.", 0.95, 0.6],
        ["narrador", "O porão está cheio de caixas de madeira. Dentro delas, tem milhares de moedas de ouro.", 1.0, 0.5],
        ["merryweather", "São trinta mil moedas de ouro. Chegaram da França e ainda estão aqui, no porão.", 1.0, 0.7],
        ["narrador", "Agora você já sabe o que o ladrão quer. Ele quer o ouro do banco.", 1.0, 0.8],
        ["holmes", "Vamos esperar no escuro. E ninguém faz barulho.", 0.9, 0.6],
        ["narrador", "Holmes cobre a lanterna. Fica tudo preto. Tão escuro que Watson não enxerga nem a própria mão.", 0.9, 0.6],
        ["narrador", "Watson só escuta a respiração dos amigos. Ele sente medo e frio, mas fica bem quietinho.", 0.85, 0.8],
        ["narrador", "Passa uma hora. Mais um pouco. Silêncio.", 0.85, 1.4],
        ["narrador", "De repente, aparece uma luzinha no chão. Uma das pedras grandes está se mexendo!", 1.05, 0.5],
        ["narrador", "A pedra sobe, e de um buraco sai uma mão branca, depois uma cabeça. É o ajudante do senhor Wilson, o Spaulding!", 1.05, 0.6],
        ["narrador", "Ele sobe para o porão e diz baixinho para um amigo que ainda está no buraco:", 0.95, 0.4],
        ["spaulding", "Tudo certo! Pegue os sacos.", 1.0, 0.5],
        ["narrador", "Nessa hora, Holmes pula e segura o braço do Spaulding com força!", 1.1, 0.4],
        ["spaulding", "Pule, Archie! Fuja!", 1.1, 0.5],
        ["narrador", "O amigo dele, o Archie, foge de volta pelo buraco. Mas não adianta: a polícia já está esperando do outro lado.", 1.05, 0.6],
        ["narrador", "O inspetor Jones prende o Spaulding. E sabe qual é o nome verdadeiro dele? John Clay, o ladrão que a polícia procura há anos!", 1.0, 0.9],
        ["narrador", "Até agora: o ajudante do senhor Wilson era um ladrão famoso, e ele foi preso no porão do banco.", 0.95, 1.2],
        ["narrador", "Mais tarde, em casa, Watson pede para Holmes explicar tudo. Holmes explica, uma pista de cada vez.", 1.0, 0.6],
        ["holmes", "Primeiro, Watson: por que o ajudante queria ganhar só metade do salário? Porque ele não queria o dinheiro. Ele queria a loja.", 1.0, 0.6],
        ["holmes", "Por que ele ficava tanto no porão? Porque estava cavando um túnel, um caminho debaixo da terra, até o banco.", 1.0, 0.6],
        ["holmes", "Por isso os joelhos da calça estavam sujos de terra. Quem cava fica de joelhos.", 1.0, 0.6],
        ["holmes", "E quando bati no chão da calçada, o som não era oco. Então o túnel não ia para a frente da loja. Ia para trás, para o banco.", 1.0, 0.8],
        ["watson", "E a Liga dos Cabeças Vermelhas?", 1.0, 0.6],
        ["holmes", "Foi tudo inventado pelo Clay e pelo amigo dele, que se fez de Duncan Ross. Eles precisavam tirar o senhor Wilson da loja, todo dia, das dez às duas.", 1.0, 0.6],
        ["holmes", "Assim, eles podiam cavar sem ninguém ouvir. Quando o túnel ficou pronto, não precisavam mais da Liga. Por isso ela acabou.", 1.0, 0.6],
        ["holmes", "E escolheram o sábado porque o banco fica fechado até segunda. O roubo só seria descoberto dois dias depois.", 1.0, 0.8],
        ["narrador", "Por que você acha que os ladrões escolheram justo o senhor Wilson?", 0.95, 2.5],
        ["narrador", "Isso mesmo: porque ele tinha a loja bem atrás do banco. E porque o cabelo vermelho dele servia de desculpa para a Liga.", 1.0, 0.8],
        ["narrador", "Para lembrar: a Liga era uma armadilha para tirar o senhor Wilson da loja, enquanto os ladrões cavavam um túnel até o ouro do banco.", 0.95, 0.8],
        ["watson", "Holmes, você é incrível.", 1.0, 0.6],
        ["narrador", "Holmes sorri. Ele está feliz, porque resolver mistérios é o que ele mais gosta de fazer.", 1.0, 0.8],
        ["narrador", "E esta aventura termina aqui. Mas Sherlock Holmes ainda vai resolver muitos outros mistérios. Até lá!", 1.0, 0.8],
    ]},
    4: {"titulo": "O chapéu e o ganso", "ambiente": "nenhum", "trechos": [
        ["narrador", "Sherlock Holmes: O Carbúnculo Azul. Parte um: O chapéu e o ganso.", 0.95, 1.2],
        ["narrador", "É Natal em Londres, há mais de cem anos. Faz muito frio, e as janelas estão cobertas de gelo.", 1.0, 0.5],
        ["narrador", "O doutor Watson vai visitar o amigo, o detetive Sherlock Holmes, para desejar feliz Natal.", 1.0, 0.5],
        ["narrador", "Holmes está sentado na poltrona. Na frente dele, numa cadeira, tem um chapéu preto, velho e todo amassado.", 1.0, 0.6],
        ["watson", "Holmes, por que você está olhando para esse chapéu velho?", 1.0, 0.6],
        ["holmes", "Porque este chapéu é uma pista, Watson. Ele chegou aqui junto com um ganso.", 1.0, 0.6],
        ["narrador", "Ganso é uma ave grande e branca, parecida com um pato. Naquele tempo, muita gente comia ganso no Natal.", 1.0, 0.8],
        ["narrador", "Holmes conta o que aconteceu. Ontem, de madrugada, um homem chamado Peterson voltava para casa.", 1.0, 0.5],
        ["narrador", "O Peterson é o porteiro de um prédio, e é muito honesto. Na rua, ele viu um senhor alto carregando um ganso nas costas.", 1.0, 0.5],
        ["narrador", "Uns rapazes começaram a brigar com esse senhor. O Peterson correu para ajudar.", 1.05, 0.5],
        ["narrador", "Mas o senhor levou um susto e fugiu correndo. E deixou para trás o chapéu e o ganso!", 1.05, 0.7],
        ["narrador", "No pé do ganso tinha um cartãozinho escrito: para a senhora de Henry Baker.", 1.0, 0.6],
        ["holmes", "Mas em Londres tem centenas de homens chamados Henry Baker. Por isso, o chapéu vai me ajudar a descobrir qual deles é o dono.", 1.0, 0.8],
        ["narrador", "Holmes pega uma lupa. Lupa é uma lente que deixa as coisas pequenas bem grandes.", 1.0, 0.5],
        ["holmes", "Veja, Watson. O chapéu é enorme. Quando eu ponho na cabeça, ele cai até o meu nariz. Então o dono tem uma cabeça muito grande.", 1.0, 0.5],
        ["holmes", "Dentro do chapéu tem pedacinhos de cabelo cinza, bem curtinhos. Então o dono é um senhor de cabelo grisalho, que cortou o cabelo há poucos dias.", 1.0, 0.5],
        ["holmes", "E tem cinco manchas de vela, de cera derretida. Então o dono sobe a escada de casa à noite com uma vela na mão. A casa dele não tem luz a gás.", 1.0, 0.8],
        ["narrador", "Viu só? Holmes olha cada pista e pensa: por que isso está assim?", 1.0, 0.8],
        ["narrador", "Até agora: um senhor chamado Henry Baker fugiu e deixou para trás um chapéu e um ganso. Holmes quer encontrar o dono.", 0.95, 1.2],
        ["narrador", "De repente, a porta se abre com força. É o Peterson, vermelho, sem fôlego!", 1.1, 0.4],
        ["peterson", "Senhor Holmes! O ganso! Olhe o que a minha mulher achou dentro do ganso!", 1.1, 0.5],
        ["narrador", "A mulher do Peterson ia preparar o ganso para o almoço de Natal. No papo da ave, ela encontrou uma coisa.", 1.0, 0.5],
        ["narrador", "Papo é uma bolsinha no pescoço das aves, onde fica a comida que elas engolem.", 1.0, 0.6],
        ["narrador", "Na mão do Peterson tem uma pedra azul, menor que um feijão. Ela brilha tanto que parece ter uma luz dentro.", 0.95, 0.8],
        ["watson", "Holmes! É o carbúnculo azul da condessa!", 1.05, 0.6],
        ["narrador", "Carbúnculo é uma pedra preciosa. E essa pedra é famosa e vale muito, muito dinheiro.", 1.0, 0.5],
        ["narrador", "Ela foi roubada há cinco dias, num hotel. A dona é uma condessa, uma senhora rica e importante.", 1.0, 0.5],
        ["narrador", "A polícia prendeu um encanador, o John Horner, porque ele tinha consertado uma coisa no quarto da condessa naquele dia.", 1.0, 0.5],
        ["narrador", "Mas o Horner chora e jura que não roubou nada.", 0.95, 0.8],
        ["holmes", "Então a pedra foi roubada no hotel e apareceu dentro de um ganso. Precisamos descobrir o caminho que ela fez.", 1.0, 0.8],
        ["narrador", "Como você acha que uma pedra preciosa foi parar dentro de um ganso?", 0.95, 2.5],
        ["narrador", "Para lembrar: a pedra roubada da condessa estava dentro do ganso do senhor Henry Baker.", 0.95, 0.8],
        ["holmes", "Peterson, ponha um anúncio no jornal. Assim: Henry Baker pode buscar o chapéu e o ganso hoje, às seis e meia, na Rua Baker.", 1.0, 0.8],
        ["narrador", "Será que o senhor Henry Baker é o ladrão? Isso fica para a próxima parte. Até lá!", 1.0, 0.8],
    ]},
    5: {"titulo": "Atrás do ganso", "ambiente": "nenhum", "trechos": [
        ["narrador", "Sherlock Holmes: O Carbúnculo Azul. Parte dois: Atrás do ganso.", 0.95, 1.2],
        ["narrador", "Na parte passada, a mulher do Peterson achou uma pedra preciosa azul dentro de um ganso. A pedra tinha sido roubada de uma condessa.", 1.0, 0.5],
        ["narrador", "O ganso era de um senhor chamado Henry Baker. Holmes pôs um anúncio no jornal para chamar esse senhor.", 1.0, 0.8],
        ["narrador", "São seis e meia da tarde. Toc, toc. Alguém bate na porta.", 1.0, 0.6],
        ["narrador", "Entra um senhor grandão, de cabeça enorme e cabelo grisalho. É o senhor Henry Baker, igualzinho ao que Holmes descobriu pelo chapéu.", 1.0, 0.6],
        ["baker", "Boa noite. Vim buscar o meu chapéu e o meu ganso.", 0.95, 0.5],
        ["holmes", "Aqui está o seu chapéu. O ganso, nós tivemos que comer, porque ia estragar. Mas compramos outro ganso para o senhor, bem gordo.", 1.0, 0.5],
        ["holmes", "Se quiser, também guardamos as penas e o papo do seu ganso antigo.", 1.0, 0.6],
        ["baker", "Não, obrigado. Para que eu ia querer isso? O ganso novo está ótimo.", 1.0, 0.7],
        ["narrador", "Holmes fica feliz. Sabe por quê?", 0.95, 0.8],
        ["narrador", "Porque, se o senhor Baker soubesse da pedra, ele ia querer o papo do ganso antigo. Então ele não é o ladrão!", 1.0, 0.6],
        ["holmes", "Senhor Baker, onde o senhor comprou aquele ganso?", 1.0, 0.5],
        ["baker", "Na pensão Alpha, perto do museu. O dono da pensão faz um clube: a gente paga um pouquinho toda semana e ganha um ganso no Natal.", 1.0, 0.8],
        ["narrador", "O senhor Baker agradece e vai embora com o ganso novo, todo contente.", 1.0, 0.8],
        ["narrador", "Até agora: o senhor Baker não é o ladrão. O ganso dele veio da pensão Alpha. Holmes quer seguir o caminho do ganso para trás.", 0.95, 1.2],
        ["narrador", "Holmes e Watson vestem os casacos e saem na noite gelada. Os passos deles fazem barulho na calçada: tec, tec, tec.", 1.0, 0.6],
        ["narrador", "Na pensão Alpha, o dono explica que comprou os gansos de um vendedor do mercado, chamado Breckinridge.", 1.0, 0.6],
        ["narrador", "Então Holmes e Watson vão ao mercado. A barraca do Breckinridge está quase fechando.", 1.0, 0.5],
        ["holmes", "Boa noite. De quem o senhor comprou os gansos que vendeu para a pensão Alpha?", 1.0, 0.5],
        ["breckinridge", "De novo essa pergunta? Todo mundo quer saber dos meus gansos! Não vou contar nada!", 1.05, 0.6],
        ["narrador", "O Breckinridge está bravo. Holmes olha para ele e vê um jornal de apostas no bolso do casaco.", 1.0, 0.5],
        ["narrador", "Aposta é quando duas pessoas discordam e combinam: quem estiver errado paga uma moeda para o outro.", 1.0, 0.6],
        ["narrador", "Holmes pensa: quem lê jornal de apostas adora apostar. Então, em vez de perguntar, ele faz uma aposta.", 1.0, 0.6],
        ["holmes", "Eu aposto uma moeda que aqueles gansos vieram de uma fazenda, lá no campo.", 1.0, 0.5],
        ["breckinridge", "Pois vai perder! Eles vieram aqui da cidade. Eu vou provar!", 1.05, 0.6],
        ["narrador", "O Breckinridge pega um caderno grande e lê em voz alta:", 1.0, 0.4],
        ["breckinridge", "Vinte e quatro gansos, da senhora Oakshott, da Rua Brixton. Vendidos para a pensão Alpha.", 1.0, 0.6],
        ["narrador", "Holmes paga a moeda e faz cara de quem perdeu. Mas, por dentro, está feliz: agora ele sabe de onde veio o ganso!", 1.0, 0.8],
        ["narrador", "Por que Holmes ficou feliz, mesmo perdendo a aposta?", 0.95, 2.5],
        ["narrador", "Para lembrar: o ganso foi da senhora Oakshott para o mercado, do mercado para a pensão Alpha, e da pensão para o senhor Baker.", 0.95, 0.8],
        ["narrador", "Quando Holmes e Watson estão saindo, escutam uma voz nervosa na barraca. É um homem baixinho, perguntando:", 0.95, 0.4],
        ["narrador", "Para onde foram os gansos da senhora Oakshott?", 0.9, 1.4],
        ["narrador", "Quem é esse homem? E por que ele quer tanto aqueles gansos? Isso fica para a próxima parte. Até lá!", 1.0, 0.8],
    ]},
    6: {"titulo": "O segredo do ganso", "ambiente": "nenhum", "trechos": [
        ["narrador", "Sherlock Holmes: O Carbúnculo Azul. Parte três: O segredo do ganso.", 0.95, 1.2],
        ["narrador", "Na parte passada, Holmes descobriu que o ganso com a pedra veio da senhora Oakshott.", 1.0, 0.5],
        ["narrador", "E, no mercado, um homem nervoso perguntava pelos gansos dela.", 1.0, 0.8],
        ["narrador", "Holmes chega perto do homem. Ele é baixinho, magro e está tremendo.", 1.0, 0.5],
        ["holmes", "Desculpe. Eu ouvi a sua pergunta. Acho que posso ajudar o senhor a achar esse ganso.", 1.0, 0.5],
        ["ryder", "O senhor sabe onde ele está? Por favor, me diga!", 1.05, 0.6],
        ["narrador", "Holmes leva o homem até a casa dele, na Rua Baker. Lá, na sala quentinha, pergunta o nome dele.", 1.0, 0.5],
        ["ryder", "Meu nome é... James Ryder. Eu trabalho no hotel.", 0.95, 0.6],
        ["narrador", "No hotel! O mesmo hotel de onde a pedra foi roubada.", 0.95, 0.8],
        ["narrador", "Holmes abre a mão devagar. Ali está a pedra azul, brilhando.", 0.9, 0.6],
        ["holmes", "É isto que o senhor procura, Ryder?", 0.9, 0.8],
        ["narrador", "O Ryder fica branco de susto. Quase cai no chão.", 0.95, 0.6],
        ["holmes", "Eu já sei de tudo, Ryder. Foi o senhor que roubou a pedra da condessa. E deixou a culpa cair no encanador, o Horner.", 1.0, 0.8],
        ["narrador", "O Ryder começa a chorar. Ele está com medo e com vergonha. E conta a história toda.", 0.95, 0.6],
        ["ryder", "Depois do roubo, eu fiquei com medo de a polícia me revistar. Então fui para a casa da minha irmã, a senhora Oakshott. Ela cria gansos.", 1.0, 0.5],
        ["ryder", "Tive uma ideia: esconder a pedra dentro de um ganso. Peguei um ganso branco, de rabo listrado, e fiz ele engolir a pedra.", 1.0, 0.5],
        ["ryder", "Mas o ganso se soltou e correu para o meio dos outros. Aí a minha irmã apareceu.", 1.0, 0.4],
        ["maggie", "Jem, o que você está fazendo com esse ganso?", 1.0, 0.5],
        ["ryder", "Eu disse que queria aquele ganso de presente de Natal. O branco, de rabo listrado.", 1.0, 0.6],
        ["narrador", "Ele levou o ganso para casa. Mas, quando abriu o papo, não tinha pedra nenhuma!", 1.05, 0.6],
        ["ryder", "Voltei correndo. A minha irmã disse que tinha dois gansos brancos de rabo listrado, iguaizinhos. E o outro já tinha ido para o mercado.", 1.0, 0.8],
        ["narrador", "Até agora: o Ryder escondeu a pedra num ganso, mas levou o ganso errado. O ganso certo foi para o mercado, depois para o senhor Baker, e depois para o Peterson.", 0.95, 1.2],
        ["ryder", "Por favor, senhor Holmes, me perdoe. Eu nunca fiz nada errado antes. Nunca mais vou fazer.", 0.95, 1.0],
        ["narrador", "Se você fosse o Holmes, o que você faria com o Ryder?", 0.95, 2.5],
        ["narrador", "Holmes fica quieto, pensando. Depois levanta e abre a porta.", 0.9, 0.6],
        ["holmes", "Vá embora, Ryder.", 0.9, 0.6],
        ["narrador", "O Ryder sai correndo escada abaixo. Tum, tum, tum, e a porta da rua bate.", 1.05, 0.8],
        ["watson", "Holmes, você deixou o ladrão ir embora?", 1.0, 0.6],
        ["holmes", "Deixei, Watson. Sem o Ryder para acusar, o encanador Horner vai ser solto, porque ele não fez nada.", 1.0, 0.5],
        ["holmes", "E o Ryder levou um susto tão grande que não vai mais roubar. Além disso, é Natal. Natal é tempo de perdoar.", 1.0, 0.8],
        ["narrador", "E a pedra azul? Holmes guarda a pedra para devolver à condessa.", 1.0, 0.6],
        ["narrador", "Para lembrar: a pedra foi roubada pelo Ryder, escondida num ganso, e encontrada por Holmes seguindo o caminho do ganso para trás.", 0.95, 0.8],
        ["narrador", "O encanador Horner vai para casa, livre. E Holmes fica feliz, porque resolveu mais um mistério.", 1.0, 0.8],
        ["narrador", "Mas Sherlock Holmes ainda tem muitas aventuras pela frente. Até lá!", 1.0, 0.8],
    ]},
}

for n, c in CAPITULOS.items():
    palavras = sum(len(t[1].split()) for t in c["trechos"])
    print(f"Capítulo {n}, {c['titulo']}: {len(c['trechos'])} trechos, cerca de "
          f"{palavras / 130 + sum(t[3] for t in c['trechos']) / 60:.1f} min.")

In [ ]:
# ── 6. Criar as vozes e ouvir (troque a semente de quem não ficou bom) ──────
import hashlib, os, shutil, requests
from IPython.display import Audio, display

def gerar(texto, destino, **campos):
    arquivos = campos.pop("files", None)
    dados = {"text": texto, "language": "pt", "effect_preset": "raw", **campos}
    r = requests.post(f"{BASE}/generate", data=dados, files=arquivos, timeout=1800)
    if r.status_code != 200:
        raise SystemExit(f"Erro {r.status_code} em /generate: {r.text[:500]}\nLog: {LOG_PATH}")
    with open(destino, "wb") as f:
        f.write(r.content)

def chave(*partes):
    return hashlib.sha1(repr(partes).encode()).hexdigest()[:8]

REF = {}   # personagem -> (wav de referência, texto)
for nome, v in VOZES.items():
    # A fala mais longa do personagem no primeiro capítulo em que ele aparece.
    texto = next(max(falas, key=len) for _, c in sorted(CAPITULOS.items())
                 if (falas := [t[1] for t in c["trechos"] if t[0] == nome]))
    arq = f"{nome}_{chave(v, texto)}.wav"
    wav = os.path.join(PASTA_VOZES, arq)
    antigo = os.path.join(LIVRO, "cap01", "vozes", arq)   # versão do primeiro teste
    if not os.path.exists(wav) and os.path.exists(antigo):
        shutil.copy(antigo, wav)
    if not os.path.exists(wav):
        if v.get("amostra"):
            with open(v["amostra"], "rb") as a:
                gerar(texto, wav, seed=v["semente"], files={"ref_audio": ("amostra.wav", a)})
        else:
            gerar(texto, wav, instruct=v["voz"], seed=v["semente"])
    REF[nome] = (wav, texto)
    print(f"{nome} ({v['voz']}, semente {v['semente']}): {texto}")
    display(Audio(wav))

In [ ]:
# ── 7. Narrar e montar cada capítulo (A: só voz; B: vinheta, sons leves, riacho) 
import os, subprocess
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

def narrar(n):
    pasta = os.path.join(LIVRO, f"cap{n:02d}", "trechos")
    os.makedirs(pasta, exist_ok=True)
    arquivos = []
    for i, t in enumerate(CAPITULOS[n]["trechos"], 1):
        quem, texto, vel, pausa = t[:4]
        efeito = t[4] if len(t) > 4 else None
        ref_wav, ref_texto = REF[quem]
        wav = os.path.join(pasta, f"{i:03d}_{quem}_{chave(os.path.basename(ref_wav), texto, vel)}.wav")
        if not os.path.exists(wav):
            with open(ref_wav, "rb") as ref:
                gerar(texto, wav, seed=VOZES[quem]["semente"], ref_text=ref_texto, speed=vel,
                      files={"ref_audio": ("ref.wav", ref, "audio/wav")})
            print(f"{n}.{i:03d} {quem}: {texto}")
        arquivos.append((wav, pausa, efeito))
    return arquivos

def tom(freq, dur, vol, sr):
    t = np.arange(int(dur * sr)) / sr
    return (vol * np.sin(2 * np.pi * freq * t) * np.exp(-3 * t)).astype("float32")

def vinheta(sr):
    # Três notas suaves (dó, mi, sol), sem ataque forte.
    v = np.zeros(int(2.2 * sr), dtype="float32")
    for k, f in enumerate([523.25, 659.25, 783.99]):
        nota = tom(f, 1.4, 0.12, sr)
        ini = int(k * 0.35 * sr)
        v[ini:ini + len(nota)] += nota
    return v

def riacho(n, sr):
    # Ruído suave filtrado, que lembra água correndo, bem baixo e com entrada lenta.
    rng = np.random.default_rng(7)
    r = np.convolve(rng.standard_normal(n + 400).astype("float32"),
                    np.ones(40, dtype="float32") / 40, mode="valid")[:n]
    r *= 0.01 * (1 + 0.3 * np.sin(2 * np.pi * 0.15 * np.arange(n) / sr))
    fade = min(n // 2, 4 * sr)
    r[:fade] *= np.linspace(0, 1, fade)
    r[-fade:] *= np.linspace(1, 0, fade)
    return r

# ── Sonoplastia leve (só na versão B) ────────────────────────────────────────
# Sete sons curtos, sintetizados aqui mesmo, sempre iguais (mesmo som = mesmo sentido).
# Entram só nas pausas, nunca por cima da fala; ataque suave; pico baixo.

def _env(n, sr, ataque=0.03, queda=4.0):
    t = np.arange(n) / sr
    a = np.clip(t / ataque, 0, 1)
    return (a * np.exp(-queda * t)).astype("float32")

def _nota(f, dur, vol, sr, queda=3.0):
    t = np.arange(int(dur * sr)) / sr
    som = np.sin(2 * np.pi * f * t) + 0.25 * np.sin(2 * np.pi * 2 * f * t)
    return (vol * som * _env(len(t), sr, 0.02, queda)).astype("float32")

def _suave(x, k):
    return np.convolve(x, np.ones(k, dtype="float32") / k, mode="same").astype("float32")

def _junta(sr, dur, pecas):
    out = np.zeros(int(dur * sr), dtype="float32")
    for ini, p in pecas:
        i = int(ini * sr)
        p = p[:len(out) - i]
        out[i:i + len(p)] += p
    return out

def ef_transicao(sr):   # troca de cena: duas notas que descem (sol, dó)
    return _junta(sr, 1.3, [(0, _nota(783.99, 1.0, 0.07, sr)), (0.3, _nota(523.25, 1.0, 0.07, sr))])

def ef_magia(sr):       # encanto, sonho, feitiço: quatro notas agudas subindo
    fs = [1046.5, 1318.5, 1568.0, 2093.0]
    return _junta(sr, 1.5, [(k * 0.12, _nota(f, 0.9, 0.035, sr, 4.0)) for k, f in enumerate(fs)])

def ef_agua(sr):        # mergulho / água: ruído abafado + duas bolhas
    rng = np.random.default_rng(11)
    n = int(0.6 * sr)
    chua = _suave(rng.standard_normal(n).astype("float32"), 60) * _env(n, sr, 0.04, 6.0) * 0.6
    def bolha(f0):
        t = np.arange(int(0.12 * sr)) / sr
        return (0.04 * np.sin(2 * np.pi * (f0 + 900 * t) * t) * _env(len(t), sr, 0.01, 20)).astype("float32")
    return _junta(sr, 0.9, [(0, chua), (0.35, bolha(500)), (0.55, bolha(650))])

def ef_galope(sr):      # cavalinho: pac, pac, pac (quatro batidinhas de madeira)
    def pac(f):
        t = np.arange(int(0.08 * sr)) / sr
        return (0.09 * np.sin(2 * np.pi * f * t) * _env(len(t), sr, 0.003, 45)).astype("float32")
    return _junta(sr, 1.2, [(k * 0.25, pac(700 if k % 2 else 600)) for k in range(4)])

def ef_porta(sr):       # batida na porta: toc, toc, toc (grave e abafado)
    def toc():
        t = np.arange(int(0.12 * sr)) / sr
        return (0.12 * np.sin(2 * np.pi * 160 * t) * _env(len(t), sr, 0.004, 30)).astype("float32")
    return _junta(sr, 1.0, [(k * 0.28, toc()) for k in range(3)])

def ef_zumbido(sr):     # abelhas: zum-zum baixinho, entra e sai devagar
    t = np.arange(int(1.2 * sr)) / sr
    z = sum(np.sin(2 * np.pi * h * 210 * t) / h for h in (1, 2, 3))
    z *= 0.5 + 0.5 * np.sin(2 * np.pi * 9 * t)
    janela = np.sin(np.pi * t / t[-1]) ** 2
    return (0.025 * z * janela).astype("float32")

def ef_festa(sr):       # alegria, festa, final feliz: acorde curto (dó, mi, sol juntos)
    return _junta(sr, 1.4, [(0, _nota(f, 1.3, 0.04, sr, 2.5)) for f in (523.25, 659.25, 783.99)])

EFEITOS = {"transicao": ef_transicao, "magia": ef_magia, "agua": ef_agua, "galope": ef_galope,
           "porta": ef_porta, "zumbido": ef_zumbido, "festa": ef_festa}

def exportar(audio, sr, pasta, nome):
    tmp = os.path.join(pasta, nome + ".wav")
    sf.write(tmp, audio, sr)
    mp3 = os.path.join(pasta, nome + ".mp3")
    # Volume normalizado para fone e caixinha: -16 LUFS, pico máximo -1.5 dBTP.
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp,
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100",
                    "-codec:a", "libmp3lame", "-q:a", "2", mp3], check=True)
    print(f"{nome}.mp3: {len(audio) / sr / 60:.1f} min")
    display(Audio(mp3))

def montar(n, arquivos):
    partes, marcas, sr, pos = [], [], None, 0
    for wav, pausa, efeito in arquivos:
        a, s = sf.read(wav, dtype="float32", always_2d=True)
        sr = sr or s
        assert s == sr, f"taxa de amostragem diferente em {wav}"
        fala = a.mean(axis=1)
        pos += len(fala)
        if efeito:
            som = EFEITOS[efeito](sr)
            pausa = max(pausa, len(som) / sr + 0.4)   # o som cabe inteiro na pausa
            marcas.append((pos + int(0.15 * sr), som))
        partes += [fala, np.zeros(int(pausa * sr), dtype="float32")]
        pos += int(pausa * sr)
    voz = np.concatenate(partes)
    # Sons das pausas: 15 dB abaixo da fala (RMS) e pico até 1/3 do pico da fala.
    ativo = voz[np.abs(voz) > 0.01]
    rms_voz = float(np.sqrt((ativo ** 2).mean())) if len(ativo) else 0.1
    sons = np.zeros_like(voz)
    for ini, som in marcas:
        sa = som[np.abs(som) > 1e-4]
        g = rms_voz * 10 ** (-15 / 20) / float(np.sqrt((sa ** 2).mean()))
        g = min(g, np.abs(voz).max() / 3 / float(np.abs(som).max()))
        sons[ini:ini + len(som)] += (som * g)[:len(sons) - ini]
    fundo = riacho(len(voz), sr) if CAPITULOS[n].get("ambiente", "riacho") == "riacho" else 0
    um_seg = np.zeros(sr, dtype="float32")
    pasta = os.path.join(LIVRO, f"cap{n:02d}")
    exportar(np.concatenate([um_seg, voz, um_seg]), sr, pasta, f"cap{n:02d}_A_so_voz")
    exportar(np.concatenate([vinheta(sr), um_seg, voz + sons + fundo, um_seg, vinheta(sr)]),
             sr, pasta, f"cap{n:02d}_B_com_ambiente")

REFAZER = False   # True: narra de novo capítulos que já têm os dois MP3 prontos

for n in CAPITULOS_PARA_GERAR:
    print(f"\n=== Capítulo {n}: {CAPITULOS[n]['titulo']} ===")
    prontos = [os.path.join(LIVRO, f"cap{n:02d}", f"cap{n:02d}_{v}.mp3")
               for v in ("A_so_voz", "B_com_ambiente")]
    if not REFAZER and all(map(os.path.exists, prontos)):
        print("Já estava pronto (para refazer, use REFAZER = True).")
        continue
    montar(n, narrar(n))
print("\nPronto! Os arquivos estão no Drive, em", LIVRO)

# Se um trecho ficou ruim, apague o .wav dele na pasta capNN/trechos/ e rode esta célula de novo.